In [ ]:
# ============================================================
# CELDA 1: Instalación de dependencias
# ============================================================
!pip install pdfplumber requests tqdm pyarrow --quiet

# ── Silenciar loggers verbosos que frenan la extracción ──────
import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)
logging.getLogger("pdfplumber").setLevel(logging.ERROR)

print("✅ Dependencias instaladas")
print("✅ Loggers pdfminer/pdfplumber silenciados")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.0 MB/s eta 0:00:00
✅ Dependencias instaladas
✅ Loggers pdfminer/pdfplumber silenciados


In [ ]:
# ============================================================
# CELDA 2: config.py
# ============================================================

import os

# ── Años a procesar ──────────────────────────────────────────
YEARS_ZIP = list(range(2016, 2025))   # 2016-2024 → ZIPs
YEARS_PDF = [2025, 2026]              # 2025-2026 → PDFs sueltos

# ── URL base ZIP (2016-2024) ──────────────────────────────────
BASE_ZIP_URL = "https://epidemiologia.salud.gob.mx/gobmx/salud/documentos/boletin/{year}.zip"

# ── URLs directas 2025 (53 semanas) ──────────────────────────
PDF_URLS_2025 = {
    1:  "https://www.gob.mx/cms/uploads/attachment/file/967123/sem01.pdf",
    2:  "https://www.gob.mx/cms/uploads/attachment/file/968145/sem02.pdf",
    3:  "https://www.gob.mx/cms/uploads/attachment/file/970198/sem03.pdf",
    4:  "https://www.gob.mx/cms/uploads/attachment/file/975996/sem04.pdf",
    5:  "https://www.gob.mx/cms/uploads/attachment/file/977285/sem05.pdf",
    6:  "https://www.gob.mx/cms/uploads/attachment/file/978817/sem06.pdf",
    7:  "https://www.gob.mx/cms/uploads/attachment/file/980033/sem07.pdf",
    8:  "https://www.gob.mx/cms/uploads/attachment/file/981239/sem08.pdf",
    9:  "https://www.gob.mx/cms/uploads/attachment/file/982667/sem09.pdf",
    10: "https://www.gob.mx/cms/uploads/attachment/file/984225/sem10.pdf",
    11: "https://www.gob.mx/cms/uploads/attachment/file/985585/Boletin-1125.pdf",
    12: "https://www.gob.mx/cms/uploads/attachment/file/986922/sem12.pdf",
    13: "https://www.gob.mx/cms/uploads/attachment/file/988378/sem13.pdf",
    14: "https://www.gob.mx/cms/uploads/attachment/file/989948/Boletin_1425_.pdf",
    15: "https://www.gob.mx/cms/uploads/attachment/file/991106/Boletin-1525.pdf",
    16: "https://www.gob.mx/cms/uploads/attachment/file/993245/sem16.pdf",
    17: "https://www.gob.mx/cms/uploads/attachment/file/996426/sem17.pdf",
    18: "https://www.gob.mx/cms/uploads/attachment/file/995846/sem18.pdf",
    19: "https://www.gob.mx/cms/uploads/attachment/file/997140/sem19.pdf",
    20: "https://www.gob.mx/cms/uploads/attachment/file/998327/sem20.pdf",
    21: "https://www.gob.mx/cms/uploads/attachment/file/999731/sem21.pdf",
    22: "https://www.gob.mx/cms/uploads/attachment/file/1001825/sem22.pdf",
    23: "https://www.gob.mx/cms/uploads/attachment/file/1002362/sem23.pdf",
    24: "https://www.gob.mx/cms/uploads/attachment/file/1003707/Boletin-SE242025.pdf",
    25: "https://www.gob.mx/cms/uploads/attachment/file/1004825/Boletin-SE252025.pdf",
    26: "https://www.gob.mx/cms/uploads/attachment/file/1006204/sem26.pdf",
    27: "https://www.gob.mx/cms/uploads/attachment/file/1007788/sem27.pdf",
    28: "https://www.gob.mx/cms/uploads/attachment/file/1009372/sem28.pdf",
    29: "https://www.gob.mx/cms/uploads/attachment/file/1011194/sem29.pdf",
    30: "https://www.gob.mx/cms/uploads/attachment/file/1012966/sem30.pdf",
    31: "https://www.gob.mx/cms/uploads/attachment/file/1014481/sem31.pdf",
    32: "https://www.gob.mx/cms/uploads/attachment/file/1015930/sem32.pdf",
    33: "https://www.gob.mx/cms/uploads/attachment/file/1018435/sem33.pdf",
    34: "https://www.gob.mx/cms/uploads/attachment/file/1019667/sem34.pdf",
    35: "https://www.gob.mx/cms/uploads/attachment/file/1020874/sem35.pdf",
    36: "https://www.gob.mx/cms/uploads/attachment/file/1022180/Boletin2025_SE36.pdf",
    37: "https://www.gob.mx/cms/uploads/attachment/file/1023743/Boletin-SE37-2025.pdf",
    38: "https://www.gob.mx/cms/uploads/attachment/file/1025250/sem38.pdf",
    39: "https://www.gob.mx/cms/uploads/attachment/file/1026612/Boletin-SE39-2025.pdf",
    40: "https://www.gob.mx/cms/uploads/attachment/file/1028410/sem40.pdf",
    41: "https://www.gob.mx/cms/uploads/attachment/file/1030196/Boletin-4125.pdf",
    42: "https://www.gob.mx/cms/uploads/attachment/file/1032380/Boletin-4225.pdf",
    43: "https://www.gob.mx/cms/uploads/attachment/file/1034886/sem43.pdf",
    44: "https://www.gob.mx/cms/uploads/attachment/file/1036631/sem44.pdf",
    45: "https://www.gob.mx/cms/uploads/attachment/file/1038567/sem45.pdf",
    46: "https://www.gob.mx/cms/uploads/attachment/file/1039257/Bolet_n-4625.pdf",
    47: "https://www.gob.mx/cms/uploads/attachment/file/1040636/Bolet_n-4725.pdf",
    48: "https://www.gob.mx/cms/uploads/attachment/file/1041826/sem48.pdf",
    49: "https://www.gob.mx/cms/uploads/attachment/file/1043402/Bolet_n-4925.pdf",
    50: "https://www.gob.mx/cms/uploads/attachment/file/1044627/Boletin-5025.pdf",
    51: "https://www.gob.mx/cms/uploads/attachment/file/1045254/Boletin2025_SE51.pdf",
    52: "https://www.gob.mx/cms/uploads/attachment/file/1046323/Boletin-5225.pdf",
    53: "https://www.gob.mx/cms/uploads/attachment/file/1047660/Boletin-5325.pdf",
}

# ── URLs directas 2026 (12 semanas) ──────────────────────────
PDF_URLS_2026 = {
    1:  "https://www.gob.mx/cms/uploads/attachment/file/1049391/sem01.pdf",
    2:  "https://www.gob.mx/cms/uploads/attachment/file/1051875/sem02.pdf",
    3:  "https://www.gob.mx/cms/uploads/attachment/file/1054461/Boletin-0326.pdf",
    4:  "https://www.gob.mx/cms/uploads/attachment/file/1055668/sem04.pdf",
    5:  "https://www.gob.mx/cms/uploads/attachment/file/1057413/sem05.pdf",
    6:  "https://www.gob.mx/cms/uploads/attachment/file/1058788/sem06.pdf",
    7:  "https://www.gob.mx/cms/uploads/attachment/file/1064304/Boletin-0726.pdf",
    8:  "https://www.gob.mx/cms/uploads/attachment/file/1064305/Boletin-0826.pdf",
    9:  "https://www.gob.mx/cms/uploads/attachment/file/1068894/Boletin-0926.pdf",
    10: "https://www.gob.mx/cms/uploads/attachment/file/1068893/Boletin-1026.pdf",
    11: "https://www.gob.mx/cms/uploads/attachment/file/1068895/Boletin-1126.pdf",
    12: "https://www.gob.mx/cms/uploads/attachment/file/1069714/Boletin-1226.pdf",
}

# Dict unificado accesible por año
PDF_URLS_BY_YEAR = {
    2025: PDF_URLS_2025,
    2026: PDF_URLS_2026,
}

# ── Directorios ───────────────────────────────────────────────
BASE_DIR      = "/content/etl_boletin"
RAW_ZIP_DIR   = os.path.join(BASE_DIR, "data", "raw", "zips")
RAW_PDF_DIR   = os.path.join(BASE_DIR, "data", "raw", "pdfs")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
LOG_DIR       = os.path.join(BASE_DIR, "logs")

# Crear directorios al cargar el config      ← AGREGADO
for _d in [RAW_ZIP_DIR, RAW_PDF_DIR, PROCESSED_DIR, LOG_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Enfermedades objetivo ─────────────────────────────────────
TARGET_DISEASES = {
    "Varicela"   : {"cie10": "B01", "tabla_keyword": "Prevenibles por Vacunación",
                    "col_keyword": "Varicela"},
    "Sarampion"  : {"cie10": "B05", "tabla_keyword": "Prevenibles por Vacunación",
                    "col_keyword": "Saramp"},
    "Rubeola"    : {"cie10": "B06", "tabla_keyword": "Prevenibles por Vacunación",
                    "col_keyword": "Rub"},
    "Escarlatina": {"cie10": "A38", "tabla_keyword": "Enfermedades Exantemáticas",
                    "col_keyword": "Escarlatina"},
    "Erisipela"  : {"cie10": "A46", "tabla_keyword": "Enfermedades Exantemáticas",
                    "col_keyword": "Erisipela"},
}

# ── DISEASE_CONFIG (usado por extractor y transformer) ────────
# ← AGREGADO — sin esto extractor.py y transformer.py fallan
DISEASE_CONFIG = {
    "Varicela": {
        "cie10"      : "B01",
        "page_kw"    : ["Varicela", "B01"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
    "Sarampion": {
        "cie10"      : "B05",
        "page_kw"    : ["Saramp", "B05"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : True,
    },
    "Rubeola": {
        "cie10"      : "B06",
        "page_kw"    : ["B06"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
    "Escarlatina": {
        "cie10"      : "A38",
        "page_kw"    : ["Escarlatina", "A38"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
    "Erisipela": {
        "cie10"      : "A46",
        "page_kw"    : ["Erisipela", "A46"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
}

# ── Catálogo de entidades (32 estados, sin TOTAL) ─────────────
ENTIDADES = [
    "Aguascalientes", "Baja California", "Baja California Sur",
    "Campeche", "Coahuila", "Colima", "Chiapas", "Chihuahua",
    "Ciudad de México", "Durango", "Guanajuato", "Guerrero",
    "Hidalgo", "Jalisco", "México", "Michoacán", "Morelos",
    "Nayarit", "Nuevo León", "Oaxaca", "Puebla", "Querétaro",
    "Quintana Roo", "San Luis Potosí", "Sinaloa", "Sonora",
    "Tabasco", "Tamaulipas", "Tlaxcala", "Veracruz",
    "Yucatán", "Zacatecas",
]

# ── Columnas del DataFrame final ──────────────────────────────
OUTPUT_COLUMNS = [
    "anio", "semana", "entidad_federativa", "enfermedad",
    "cie10", "sexo", "casos_semana", "casos_acumulado",
    "casos_acum_anterior",
]

# ── Parámetros de descarga ────────────────────────────────────
DOWNLOAD_TIMEOUT = 60
MAX_RETRIES      = 3
CHUNK_SIZE       = 1024 * 1024   # 1 MB

# ── Resumen al cargar ─────────────────────────────────────────
print("✅ config.py cargado correctamente")
print(f"   Años ZIP     : {YEARS_ZIP[0]} → {YEARS_ZIP[-1]}")
print(f"   Años PDF     : {YEARS_PDF}")
print(f"   URLs 2025    : {len(PDF_URLS_2025)} semanas")
print(f"   URLs 2026    : {len(PDF_URLS_2026)} semanas")
print(f"   Enfermedades : {list(DISEASE_CONFIG.keys())}")
print(f"   Directorios  : {BASE_DIR}")

✅ config.py cargado correctamente
   Años ZIP     : 2016 → 2024
   Años PDF     : [2025, 2026]
   URLs 2025    : 53 semanas
   URLs 2026    : 12 semanas
   Enfermedades : ['Varicela', 'Sarampion', 'Rubeola', 'Escarlatina', 'Erisipela']
   Directorios  : /content/etl_boletin


In [ ]:
# ============================================================
# CELDA 3: downloader.py
# Descarga ZIPs (2016-2024) y PDFs sueltos (2025-2026)
# ============================================================

import os
import time
import logging
import zipfile
import requests
from tqdm import tqdm
from pathlib import Path

os.makedirs(LOG_DIR, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(LOG_DIR, "download.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("downloader")


def _make_dirs():
    for d in [RAW_ZIP_DIR, RAW_PDF_DIR, PROCESSED_DIR]:
        os.makedirs(d, exist_ok=True)


def _download_file(url: str, dest_path: str, desc: str = "") -> bool:
    """Descarga con reintentos y barra de progreso."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, stream=True, timeout=DOWNLOAD_TIMEOUT)
            resp.raise_for_status()
            total = int(resp.headers.get("content-length", 0))
            with open(dest_path, "wb") as f, tqdm(
                desc=desc, total=total, unit="B",
                unit_scale=True, unit_divisor=1024, leave=False,
            ) as bar:
                for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                    f.write(chunk)
                    bar.update(len(chunk))
            logger.info(f"OK | {url}")
            return True
        except requests.exceptions.RequestException as e:
            logger.warning(f"Intento {attempt}/{MAX_RETRIES} | {url} | {e}")
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
    logger.error(f"FALLO | {url}")
    return False


def _unzip(zip_path: str, year: int) -> list:
    """
    Descomprime ZIP y retorna lista de rutas PDF.
    Maneja ZIPs con subcarpetas internas de cualquier profundidad.
    """
    year_dir = os.path.join(RAW_PDF_DIR, str(year))
    os.makedirs(year_dir, exist_ok=True)

    pdfs = []
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            # Listar contenido para diagnóstico
            members = z.namelist()
            pdf_members = [m for m in members if m.lower().endswith(".pdf")]

            if not pdf_members:
                logger.warning(f"{year}: ZIP sin PDFs — contenido: {members[:5]}")
                return []

            for member in pdf_members:
                # Extraer aplanando la estructura de carpetas
                # sem01.pdf, carpeta/sem01.pdf, a/b/sem01.pdf → todos van a year_dir
                filename = os.path.basename(member)
                if not filename:          # es una carpeta, no un archivo
                    continue

                dest = os.path.join(year_dir, filename)

                # Extraer a carpeta temporal y mover
                with z.open(member) as src, open(dest, "wb") as dst:
                    dst.write(src.read())

                pdfs.append(dest)

        logger.info(f"Descomprimido {year}: {len(pdfs)} PDFs")

    except zipfile.BadZipFile as e:
        logger.error(f"ZIP corrupto {zip_path}: {e}")
    except Exception as e:
        logger.error(f"Error descomprimiendo {zip_path}: {e}")

    return pdfs


# ─────────────────────────────────────────────────────────────
# DESCARGA DE ZIPs (2016-2024)
# ─────────────────────────────────────────────────────────────

def download_zips(years: list = None) -> dict:
    """Descarga y descomprime ZIPs. Retorna {año: [rutas PDF]}."""
    _make_dirs()
    years  = years or YEARS_ZIP
    result = {}
    print(f"\n📦 Descargando ZIPs: {years[0]} → {years[-1]}")

    for year in tqdm(years, desc="Años"):
        url      = BASE_ZIP_URL.format(year=year)
        zip_path = os.path.join(RAW_ZIP_DIR, f"{year}.zip")

        if os.path.exists(zip_path):
            print(f"   {year}: ZIP ya existe, omitiendo descarga")
            logger.info(f"SKIP | {zip_path}")
        else:
            ok = _download_file(url, zip_path, desc=f"ZIP {year}")
            if not ok:
                result[year] = []
                continue

        pdfs = _unzip(zip_path, year)
        result[year] = pdfs
        print(f"   {year}: {len(pdfs)} PDFs extraídos")

    return result


# ─────────────────────────────────────────────────────────────
# DESCARGA DE PDFs SUELTOS (2025-2026)
# ─────────────────────────────────────────────────────────────

def download_pdfs_sueltos(years: list = None) -> dict:
    """
    Descarga PDFs individuales para 2025 y 2026
    usando las URLs definidas en PDF_URLS_BY_YEAR del config.

    Nombre de archivo guardado:
      2025 → sem{semana:02d}_2025.pdf
      2026 → sem{semana:02d}_2026.pdf
    """
    _make_dirs()
    years  = years or YEARS_PDF
    result = {}

    for year in years:
        urls_year = PDF_URLS_BY_YEAR.get(year, {})

        if not urls_year:
            print(f"\n⚠️  {year}: no hay URLs configuradas en PDF_URLS_BY_YEAR")
            result[year] = []
            continue

        year_dir = os.path.join(RAW_PDF_DIR, str(year))
        os.makedirs(year_dir, exist_ok=True)

        print(f"\n📄 Descargando PDFs sueltos {year}: "
              f"{len(urls_year)} semanas configuradas")

        pdfs_ok = []
        for semana, url in tqdm(sorted(urls_year.items()),
                                desc=f"PDFs {year}"):
            # Nombre estandarizado
            filename = f"sem{semana:02d}_{year}.pdf"
            dest     = os.path.join(year_dir, filename)

            if os.path.exists(dest):
                print(f"   {year} sem{semana:02d}: ya existe, omitiendo")
                pdfs_ok.append(dest)
                continue

            ok = _download_file(url, dest, desc=f"{year} sem{semana:02d}")
            if ok:
                pdfs_ok.append(dest)
                logger.info(f"PDF descargado: {dest}")
            else:
                logger.error(f"Fallo sem{semana:02d}/{year}: {url}")

        result[year] = pdfs_ok
        print(f"   {year}: {len(pdfs_ok)}/{len(urls_year)} PDFs descargados")

    return result


# ─────────────────────────────────────────────────────────────
# INVENTARIO
# ─────────────────────────────────────────────────────────────

def get_pdf_inventory(years: list = None) -> dict:
    """Escanea RAW_PDF_DIR y retorna {año: [rutas PDF]}."""
    years = years or YEARS_ZIP + YEARS_PDF
    inventory = {}
    for year in years:
        year_dir = os.path.join(RAW_PDF_DIR, str(year))
        if os.path.isdir(year_dir):
            pdfs = sorted(str(p) for p in Path(year_dir).rglob("*.pdf"))
            inventory[year] = pdfs
        else:
            inventory[year] = []
    return inventory


def print_inventory(inventory: dict):
    """Imprime resumen del inventario."""
    print("\n📋 Inventario de PDFs:")
    print(f"{'Año':<8} {'PDFs':<8} {'Fuente'}")
    print("-" * 30)
    total = 0
    for year, pdfs in sorted(inventory.items()):
        fuente = "ZIP" if year in YEARS_ZIP else "PDF suelto"
        print(f"{year:<8} {len(pdfs):<8} {fuente}")
        total += len(pdfs)
    print("-" * 30)
    print(f"{'TOTAL':<8} {total:<8}")


print("✅ downloader.py cargado correctamente")

✅ downloader.py cargado correctamente


In [ ]:
# ============================================================
# CELDA 4: Ejecución del Módulo 1
# ============================================================

# ── PASO 1: ZIPs 2016-2024 ───────────────────────────────────
pdf_inventory_zip = download_zips(years=YEARS_ZIP)

# ── PASO 2: PDFs sueltos 2025-2026 ───────────────────────────
pdf_inventory_pdf = download_pdfs_sueltos(years=YEARS_PDF)

# ── PASO 3: Inventario completo ──────────────────────────────
inventory = get_pdf_inventory(years=YEARS_ZIP + YEARS_PDF)
print_inventory(inventory)

# ── PASO 4: Diagnóstico detallado ────────────────────────────
print("\n🔍 Diagnóstico por año:")
for year in YEARS_ZIP:
    zip_path = os.path.join(RAW_ZIP_DIR, f"{year}.zip")
    n_pdfs   = len(inventory.get(year, []))
    zip_mb   = os.path.getsize(zip_path) / 1_048_576 if os.path.exists(zip_path) else 0
    status   = "✅" if n_pdfs >= 50 else "⚠️ " if n_pdfs > 0 else "❌"
    print(f"  {status} {year}: ZIP={zip_mb:.0f}MB | PDFs={n_pdfs}")

for year in YEARS_PDF:
    n_pdfs = len(inventory.get(year, []))
    status = "✅" if n_pdfs > 0 else "⚠️ "
    print(f"  {status} {year}: PDFs={n_pdfs}/{len(PDF_URLS_BY_YEAR.get(year,{}))}")

# ── PASO 5: Re-descomprimir años fallidos ─────────────────────
años_fallidos = [
    y for y in YEARS_ZIP
    if len(inventory.get(y, [])) == 0
    and os.path.exists(os.path.join(RAW_ZIP_DIR, f"{y}.zip"))
]

if años_fallidos:
    print(f"\n🔧 Re-descomprimiendo años fallidos: {años_fallidos}")
    for year in años_fallidos:
        zip_path = os.path.join(RAW_ZIP_DIR, f"{year}.zip")
        pdfs = _unzip(zip_path, year)
        inventory[year] = pdfs
        print(f"   {year}: {len(pdfs)} PDFs extraídos")
    print_inventory(inventory)
else:
    print("\n✅ Todos los ZIPs descomprimidos correctamente")

print("✅ Ejecución del Módulo 1 finalizada")


📦 Descargando ZIPs: 2016 → 2024


Años:   0%|          | 0/9 [00:00<?, ?it/s]
ZIP 2016:   0%|          | 0.00/33.9M [00:00<?, ?B/s]
ZIP 2016:   3%|▎         | 1.00M/33.9M [00:00<00:07, 4.41MB/s]
ZIP 2016:  12%|█▏        | 4.00M/33.9M [00:00<00:02, 12.0MB/s]
ZIP 2016:  21%|██        | 7.00M/33.9M [00:00<00:01, 16.2MB/s]
ZIP 2016:  27%|██▋       | 9.00M/33.9M [00:00<00:01, 14.4MB/s]
ZIP 2016:  35%|███▌      | 12.0M/33.9M [00:00<00:01, 15.7MB/s]
ZIP 2016:  41%|████▏     | 14.0M/33.9M [00:01<00:01, 15.3MB/s]
ZIP 2016:  47%|████▋     | 16.0M/33.9M [00:01<00:01, 15.2MB/s]
ZIP 2016:  53%|█████▎    | 18.0M/33.9M [00:01<00:01, 15.6MB/s]
ZIP 2016:  59%|█████▉    | 20.0M/33.9M [00:01<00:00, 16.0MB/s]
ZIP 2016:  65%|██████▍   | 22.0M/33.9M [00:01<00:00, 16.2MB/s]
ZIP 2016:  71%|███████   | 24.0M/33.9M [00:01<00:00, 16.0MB/s]
ZIP 2016:  80%|███████▉  | 27.0M/33.9M [00:01<00:00, 17.1MB/s]
ZIP 2016:  88%|████████▊ | 30.0M/33.9M [00:02<00:00, 17.9MB/s]
ZIP 2016:  94%|█████████▍| 32.0M/33.9M [00:02<00:00, 18.4MB/s]
ZIP 2016: 100%|█████

   2016: 52 PDFs extraídos



ZIP 2017:   0%|          | 0.00/35.7M [00:00<?, ?B/s]
ZIP 2017:   3%|▎         | 1.00M/35.7M [00:00<00:08, 4.18MB/s]
ZIP 2017:  11%|█         | 4.00M/35.7M [00:00<00:02, 11.6MB/s]
ZIP 2017:  17%|█▋        | 6.00M/35.7M [00:00<00:02, 14.2MB/s]
ZIP 2017:  22%|██▏       | 8.00M/35.7M [00:00<00:02, 11.0MB/s]
ZIP 2017:  28%|██▊       | 10.0M/35.7M [00:00<00:02, 12.8MB/s]
ZIP 2017:  34%|███▎      | 12.0M/35.7M [00:00<00:01, 14.5MB/s]
ZIP 2017:  39%|███▉      | 14.0M/35.7M [00:01<00:01, 15.6MB/s]
ZIP 2017:  45%|████▍     | 16.0M/35.7M [00:01<00:01, 16.8MB/s]
ZIP 2017:  53%|█████▎    | 19.0M/35.7M [00:01<00:00, 18.4MB/s]
ZIP 2017:  59%|█████▊    | 21.0M/35.7M [00:01<00:00, 19.0MB/s]
ZIP 2017:  67%|██████▋   | 24.0M/35.7M [00:01<00:00, 20.0MB/s]
ZIP 2017:  73%|███████▎  | 26.0M/35.7M [00:01<00:00, 19.9MB/s]
ZIP 2017:  81%|████████  | 29.0M/35.7M [00:01<00:00, 20.9MB/s]
ZIP 2017:  87%|████████▋ | 31.0M/35.7M [00:02<00:00, 16.4MB/s]
ZIP 2017:  95%|█████████▌| 34.0M/35.7M [00:02<00:00, 19.7MB/s]


   2017: 52 PDFs extraídos


...

   2024: 52 PDFs extraídos

📄 Descargando PDFs sueltos 2025: 53 semanas configuradas


PDFs 2025:   0%|          | 0/53 [00:00<?, ?it/s]
2025 sem01:   0%|          | 0.00/1.94M [00:00<?, ?B/s]
PDFs 2025:   2%|▏         | 1/53 [00:00<00:11,  4.43it/s]
2025 sem02:   0%|          | 0.00/2.02M [00:00<?, ?B/s]
PDFs 2025:   4%|▍         | 2/53 [00:00<00:09,  5.27it/s]
2025 sem03:   0%|          | 0.00/2.13M [00:00<?, ?B/s]
PDFs 2025:   6%|▌         | 3/53 [00:00<00:09,  5.24it/s]
2025 sem04:   0%|          | 0.00/2.02M [00:00<?, ?B/s]
PDFs 2025:   8%|▊         | 4/53 [00:00<00:08,  5.58it/s]
2025 sem05:   0%|          | 0.00/2.01M [00:00<?, ?B/s]
PDFs 2025:   9%|▉         | 5/53 [00:00<00:08,  5.99it/s]
2025 sem06:   0%|          | 0.00/2.14M [00:00<?, ?B/s]
PDFs 2025:  11%|█▏        | 6/53 [00:01<00:07,  6.15it/s]
2025 sem07:   0%|          | 0.00/1.77M [00:00<?, ?B/s]
PDFs 2025:  13%|█▎        | 7/53 [00:01<00:07,  6.38it/s]
2025 sem08:   0%|          | 0.00/2.09M [00:00<?, ?B/s]
2025 sem08:  96%|█████████▌| 2.00M/2.09M [00:00<00:00, 20.7MB/s]
PDFs 2025:  15%|█▌        | 8/5

   2025: 53/53 PDFs descargados

📄 Descargando PDFs sueltos 2026: 12 semanas configuradas


PDFs 2026:   0%|          | 0/12 [00:00<?, ?it/s]
2026 sem01:   0%|          | 0.00/2.35M [00:00<?, ?B/s]
PDFs 2026:   8%|▊         | 1/12 [00:00<00:01,  7.06it/s]
2026 sem02:   0%|          | 0.00/2.09M [00:00<?, ?B/s]
2026 sem02:  48%|████▊     | 1.00M/2.09M [00:00<00:00, 7.15MB/s]
PDFs 2026:  17%|█▋        | 2/12 [00:00<00:02,  4.71it/s]
2026 sem03:   0%|          | 0.00/2.04M [00:00<?, ?B/s]
PDFs 2026:  25%|██▌       | 3/12 [00:00<00:01,  5.26it/s]
2026 sem04:   0%|          | 0.00/2.28M [00:00<?, ?B/s]
PDFs 2026:  33%|███▎      | 4/12 [00:00<00:01,  5.80it/s]
2026 sem05:   0%|          | 0.00/2.36M [00:00<?, ?B/s]
2026 sem05:  42%|████▏     | 1.00M/2.36M [00:00<00:00, 7.45MB/s]
PDFs 2026:  42%|████▏     | 5/12 [00:01<00:01,  4.66it/s]
2026 sem06:   0%|          | 0.00/2.43M [00:00<?, ?B/s]
PDFs 2026:  50%|█████     | 6/12 [00:01<00:01,  5.07it/s]
2026 sem07:   0%|          | 0.00/3.48M [00:00<?, ?B/s]
PDFs 2026:  58%|█████▊    | 7/12 [00:01<00:00,  5.49it/s]
2026 sem08:   0%|     

   2026: 12/12 PDFs descargados

📋 Inventario de PDFs:
Año      PDFs     Fuente
------------------------------
2016     52       ZIP
2017     52       ZIP
2018     52       ZIP
2019     52       ZIP
2020     53       ZIP
2021     52       ZIP
2022     52       ZIP
2023     53       ZIP
2024     52       ZIP
2025     53       PDF suelto
2026     12       PDF suelto
------------------------------
TOTAL    535     

🔍 Diagnóstico por año:
  ✅ 2016: ZIP=34MB | PDFs=52
  ✅ 2017: ZIP=36MB | PDFs=52
  ✅ 2018: ZIP=36MB | PDFs=52
  ✅ 2019: ZIP=47MB | PDFs=52
  ✅ 2020: ZIP=54MB | PDFs=53
  ✅ 2021: ZIP=48MB | PDFs=52
  ✅ 2022: ZIP=98MB | PDFs=52
  ✅ 2023: ZIP=127MB | PDFs=53
  ✅ 2024: ZIP=105MB | PDFs=52
  ✅ 2025: PDFs=53/53
  ✅ 2026: PDFs=12/12

✅ Todos los ZIPs descomprimidos correctamente
✅ Ejecución del Módulo 1 finalizada


# **Modulo 2**

In [ ]:
# ============================================================
# CELDA 5: extractor.py  ── v4
# Igual que v3 excepto _detect_year y _detect_week corregidos
# ============================================================

import re
import logging
import pdfplumber
import pandas as pd
from pathlib import Path

logger = logging.getLogger("extractor")

ENTIDADES_SORTED = sorted(ENTIDADES, key=len, reverse=True)

DISEASE_CONFIG = {
    "Varicela": {
        "cie10"      : "B01",
        "page_kw"    : ["Varicela", "B01"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
    "Sarampion": {
        "cie10"      : "B05",
        "page_kw"    : ["Saramp", "B05"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : True,
    },
    "Rubeola": {
        "cie10"      : "B06",
        "page_kw"    : ["B06"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
    "Escarlatina": {
        "cie10"      : "A38",
        "page_kw"    : ["Escarlatina", "A38"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
    "Erisipela": {
        "cie10"      : "A46",
        "page_kw"    : ["Erisipela", "A46"],
        "require_kw" : ["ENTIDAD", "FEDERATIVA"],
        "optional"   : False,
    },
}

# ─────────────────────────────────────────────────────────────
# UTILS
# ─────────────────────────────────────────────────────────────

def _parse_num(s) -> int:
    if s is None:
        return 0
    s = (str(s).strip()
         .replace('\u202f', '').replace('\xa0', '').replace(' ', ''))
    if s in ('-', '', 'None', 'n.d.', 'n.a.', 's.n.'):
        return 0
    try:
        return int(s)
    except ValueError:
        return 0


def _detect_year(text: str, filename: str) -> int:
    """
    Formatos reales en página 0:
      2020: "Número 23 | Volumen 37 | Semana 23 | Del 31 de mayo al 6 de junio del 2020"
      2025: "Número 52 / Volumen 42 / Semana 52 / Del 21 al 27 de diciembre del 2025"
    Estrategia: buscar primer 20XX en el texto completo.
    Fallback: buscar en nombre de archivo.
    """
    years = re.findall(r'\b(20\d{2})\b', text)
    if years:
        return int(years[0])
    m = re.search(r'\b(20\d{2})\b', filename)
    return int(m.group(1)) if m else 0


def _detect_week(text: str, filename: str) -> int:
    """
    Busca 'Semana NN' en texto. Fallback: nombre de archivo.
    """
    m = re.search(r'[Ss]emana\s+(\d{1,2})\b', text)
    if m:
        return int(m.group(1))
    m = re.search(r'[Ss]em[_\-]?(\d{1,2})', filename, re.I)
    return int(m.group(1)) if m else 0


def _detect_sexo(table: list) -> list:
    if len(table) >= 4 and table[3]:
        row3 = [c for c in table[3] if c]
        if 'F' in row3:
            return ['M', 'F']
    return ['H', 'M']


def _find_header_idx(table: list, cie10: str):
    if not table:
        return None
    cie10_norm = cie10.upper().replace('ª', 'A')
    for idx, cell in enumerate(table[0]):
        if not cell:
            continue
        if cie10_norm in str(cell).upper().replace('ª', 'A'):
            return idx
    return None


def _page_matches(text: str, cfg: dict) -> bool:
    return (any(kw in text for kw in cfg["page_kw"]) and
            all(kw in text for kw in cfg["require_kw"]))


# ─────────────────────────────────────────────────────────────
# PARSER UNIFICADO
# ─────────────────────────────────────────────────────────────

def _parse_block(table, header_idx, enfermedad, cie10,
                 anio, semana, sexo_labels) -> list:
    DATA_ROW = 4
    if len(table) <= DATA_ROW:
        logger.warning(f"Tabla incompleta ({len(table)} filas) para {cie10}")
        return []

    row    = table[DATA_ROW]
    n_cols = len(row)

    idx_hm     = header_idx + 1
    idx_ant    = header_idx + 3
    is_primary = (header_idx == 1)
    idx_sem2   = header_idx if not is_primary else None

    for idx, name in [(idx_hm, "HM"), (idx_ant, "ant")]:
        if idx >= n_cols:
            logger.warning(
                f"{cie10}: índice {name}={idx} fuera de rango "
                f"(tabla tiene {n_cols} cols, header_idx={header_idx})"
            )
            return []

    col0     = row[0]          or ""
    col_hm   = row[idx_hm]    or ""
    col_ant  = row[idx_ant]   or ""
    col_sem2 = (row[idx_sem2] or "") if idx_sem2 is not None else None

    lines0     = [l.strip() for l in str(col0).split('\n')    if l.strip()]
    lines_hm   = [l.strip() for l in str(col_hm).split('\n') if l.strip()]
    lines_ant  = [l.strip() for l in str(col_ant).split('\n')if l.strip()]
    lines_sem2 = (
        [l.strip() for l in str(col_sem2).split('\n') if l.strip()]
        if col_sem2 is not None else None
    )

    records = []

    for i, line in enumerate(lines0):
        entidad = None
        sem_raw = ""
        for e in ENTIDADES_SORTED:
            if line.startswith(e):
                entidad = e
                sem_raw = line[len(e):].strip()
                break
        if entidad is None:
            continue

        if lines_sem2 is not None:
            casos_sem = _parse_num(lines_sem2[i]) if i < len(lines_sem2) else 0
        else:
            casos_sem = _parse_num(sem_raw)

        if i < len(lines_hm):
            parts    = lines_hm[i].split()
            acum_sx1 = _parse_num(parts[0]) if len(parts) > 0 else 0
            acum_sx2 = _parse_num(parts[1]) if len(parts) > 1 else 0
        else:
            acum_sx1 = acum_sx2 = 0

        acum_ant = _parse_num(lines_ant[i]) if i < len(lines_ant) else 0

        for sexo, acum_val in zip(sexo_labels, [acum_sx1, acum_sx2]):
            records.append({
                "anio"               : anio,
                "semana"             : semana,
                "entidad_federativa" : entidad,
                "enfermedad"         : enfermedad,
                "cie10"              : cie10,
                "sexo"               : sexo,
                "casos_semana"       : casos_sem,
                "casos_acumulado"    : acum_val,
                "casos_acum_anterior": acum_ant,
            })

    return records


# ─────────────────────────────────────────────────────────────
# EXTRACTOR PRINCIPAL
# ─────────────────────────────────────────────────────────────

def extract_pdf(pdf_path: str) -> pd.DataFrame:
    all_records = []
    pdf_name    = Path(pdf_path).name

    try:
        with pdfplumber.open(pdf_path) as pdf:
            text_p0 = pdf.pages[0].extract_text() or ""
            anio    = _detect_year(text_p0, pdf_name)
            semana  = _detect_week(text_p0, pdf_name)
            logger.info(f"Procesando {pdf_name} → Año:{anio} Semana:{semana}")

            extracted = {name: False for name in DISEASE_CONFIG}

            for page_num, page in enumerate(pdf.pages, 1):
                if all(extracted.values()):
                    break

                text = page.extract_text() or ""

                pending = [
                    name for name, cfg in DISEASE_CONFIG.items()
                    if not extracted[name] and _page_matches(text, cfg)
                ]
                if not pending:
                    continue

                tables = page.extract_tables()
                if not tables:
                    continue

                table       = tables[0]
                sexo_labels = _detect_sexo(table)

                for name in pending:
                    cfg        = DISEASE_CONFIG[name]
                    cie10      = cfg["cie10"]
                    header_idx = _find_header_idx(table, cie10)

                    if header_idx is None:
                        logger.warning(
                            f"{pdf_name} p{page_num}: CIE-10 {cie10} "
                            f"no encontrado en header"
                        )
                        continue

                    records = _parse_block(
                        table, header_idx, name, cie10,
                        anio, semana, sexo_labels
                    )

                    if records:
                        all_records.extend(records)
                        extracted[name] = True
                        logger.info(
                            f"{pdf_name} p{page_num}: {cie10} "
                            f"(header_idx={header_idx}) → {len(records)} registros"
                        )
                    else:
                        logger.warning(
                            f"{pdf_name} p{page_num}: {cie10} → 0 registros"
                        )

            missing_required = [
                n for n, v in extracted.items()
                if not v and not DISEASE_CONFIG[n].get("optional", False)
            ]
            missing_optional = [
                n for n, v in extracted.items()
                if not v and DISEASE_CONFIG[n].get("optional", False)
            ]
            if missing_required:
                logger.warning(
                    f"{pdf_name}: FALTANTES REQUERIDAS → {missing_required}"
                )
            if missing_optional:
                logger.info(
                    f"{pdf_name}: sin casos (opcional) → {missing_optional}"
                )

    except Exception as e:
        logger.error(f"Error procesando {pdf_name}: {e}", exc_info=True)
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    if not all_records:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    df = pd.DataFrame(all_records, columns=OUTPUT_COLUMNS)
    for col in ["casos_semana", "casos_acumulado", "casos_acum_anterior"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    df["anio"]   = df["anio"].astype(int)
    df["semana"] = df["semana"].astype(int)
    return df


# ─────────────────────────────────────────────────────────────
# PROCESAR MÚLTIPLES PDFs
# ─────────────────────────────────────────────────────────────

def extract_all_pdfs(inventory: dict, years: list = None) -> pd.DataFrame:
    from tqdm import tqdm
    years      = years or sorted(inventory.keys())
    frames     = []
    total_pdfs = sum(len(inventory.get(y, [])) for y in years)
    print(f"\n🔬 Extrayendo datos de {total_pdfs} PDFs...")

    with tqdm(total=total_pdfs, unit="PDF") as pbar:
        for year in years:
            year_frames = []
            for pdf_path in sorted(inventory.get(year, [])):
                df = extract_pdf(pdf_path)
                if not df.empty:
                    year_frames.append(df)
                pbar.update(1)
                pbar.set_postfix({"año": year, "ok": len(year_frames)})
            if year_frames:
                year_df = pd.concat(year_frames, ignore_index=True)
                frames.append(year_df)
                print(f"   {year}: {len(year_df):,} registros")

    if not frames:
        print("⚠️  No se extrajeron datos")
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    df_final = (pd.concat(frames, ignore_index=True)
                .sort_values(["anio", "semana", "enfermedad",
                              "entidad_federativa", "sexo"])
                .reset_index(drop=True))
    print(f"\n✅ Total: {len(df_final):,} registros")
    print(f"   Enfermedades : {sorted(df_final['enfermedad'].unique())}")
    print(f"   Años         : {sorted(df_final['anio'].unique())}")
    return df_final

def extract_all_pdfs(inventory: dict, years: list = None) -> pd.DataFrame:
    from tqdm import tqdm
    years      = years or sorted(inventory.keys())
    frames     = []
    total_pdfs = sum(len(inventory.get(y, [])) for y in years)
    print(f"\n🔬 Extrayendo datos de {total_pdfs} PDFs...")

    with tqdm(total=total_pdfs, unit="PDF") as pbar:
        for year in years:
            year_frames = []
            # ── NUEVO: eliminar rutas duplicadas en el inventario ──
            pdfs_unicos = list(dict.fromkeys(inventory.get(year, [])))
            if len(pdfs_unicos) < len(inventory.get(year, [])):
                logger.warning(
                    f"{year}: {len(inventory[year])-len(pdfs_unicos)} "
                    f"PDFs duplicados en inventario — ignorados"
                )
            for pdf_path in sorted(pdfs_unicos):
                df = extract_pdf(pdf_path)
                if not df.empty:
                    year_frames.append(df)
                pbar.update(1)
                pbar.set_postfix({"año": year, "ok": len(year_frames)})
            if year_frames:
                year_df = pd.concat(year_frames, ignore_index=True)
                frames.append(year_df)
                print(f"   {year}: {len(year_df):,} registros")

    if not frames:
        print("⚠️  No se extrajeron datos")
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    df_final = (pd.concat(frames, ignore_index=True)
                .sort_values(["anio","semana","enfermedad",
                              "entidad_federativa","sexo"])
                .reset_index(drop=True))
    print(f"\n✅ Total: {len(df_final):,} registros")
    print(f"   Enfermedades : {sorted(df_final['enfermedad'].unique())}")
    print(f"   Años         : {sorted(df_final['anio'].unique())}")
    return df_final


print("✅ extractor.py v4 cargado")

✅ extractor.py v4 cargado


In [ ]:
# ============================================================
# CELDA 6: Test extractor con sem23/2020
# ============================================================

import logging

# Limpiar handlers anteriores
for h in logging.getLogger("extractor").handlers[:]:
    logging.getLogger("extractor").removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)

df_test = extract_pdf("/content/etl_boletin/data/raw/pdfs/2020/sem23.pdf")

print(f"\nShape: {df_test.shape}")
print(f"\nRegistros por enfermedad y sexo (acumulado):")
print(df_test.groupby(["enfermedad", "sexo"])["casos_acumulado"].sum().to_string())

print(f"\nVerificación puntual contra PDF (sem23/2020):")
checks = [
    # (enfermedad, sexo, estado, expected, optional)
    ("Varicela",    "M", "Sonora",           194,  False),
    ("Varicela",    "F", "Sonora",           182,  False),
    ("Escarlatina", "M", "Chihuahua",         38,  False),
    ("Escarlatina", "F", "Ciudad de México",  73,  False),
    ("Erisipela",   "M", "Tabasco",          105,  False),
    ("Erisipela",   "F", "Tamaulipas",       214,  False),
    ("Rubeola",     "M", "Aguascalientes",     0,  False),
    ("Sarampion",   "M", "Aguascalientes",  None,  True),
]

all_ok = True
for enf, sx, estado, expected, optional in checks:
    val = df_test[
        (df_test.enfermedad == enf) &
        (df_test.sexo == sx) &
        (df_test.entidad_federativa == estado)
    ]["casos_acumulado"].values

    got = val[0] if len(val) else None

    if optional and expected is None:
        status = "✅"
        nota   = "(sin casos en esta semana — correcto)"
    elif got == expected:
        status = "✅"
        nota   = ""
    else:
        status = "❌"
        nota   = ""
        all_ok = False

    print(f"  {status} {enf} {sx} {estado}: "
          f"esperado={expected}, obtenido={got} {nota}")

print(f"\n{'✅ TODOS LOS CHECKS PASARON' if all_ok else '❌ HAY CHECKS FALLIDOS'}")


Shape: (256, 9)

Registros por enfermedad y sexo (acumulado):
enfermedad   sexo
Erisipela    F        1709
             M        2036
Escarlatina  F         426
             M         486
Rubeola      F           0
             M           0
Varicela     F       10917
             M       10409

Verificación puntual contra PDF (sem23/2020):
  ✅ Varicela M Sonora: esperado=194, obtenido=194 
  ✅ Varicela F Sonora: esperado=182, obtenido=182 
  ✅ Escarlatina M Chihuahua: esperado=38, obtenido=38 
  ✅ Escarlatina F Ciudad de México: esperado=73, obtenido=73 
  ✅ Erisipela M Tabasco: esperado=105, obtenido=105 
  ✅ Erisipela F Tamaulipas: esperado=214, obtenido=214 
  ✅ Rubeola M Aguascalientes: esperado=0, obtenido=0 
  ✅ Sarampion M Aguascalientes: esperado=None, obtenido=None (sin casos en esta semana — correcto)

✅ TODOS LOS CHECKS PASARON


# **Modulo 3**

In [ ]:
# ============================================================
# CELDA 7: transformer.py  ── v3
# ============================================================

import os
import numpy as np
import logging
import pandas as pd
import datetime as _dt

logger = logging.getLogger("transformer")

DISEASE_CIE10 = {
    name: cfg["cie10"]
    for name, cfg in DISEASE_CONFIG.items()
}


# ─────────────────────────────────────────────────────────────
# PASO 1 — COMPLETAR COMBINACIONES FALTANTES
# ─────────────────────────────────────────────────────────────

def _fill_missing_combinations(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    # Deduplicar antes de set_index
    key_cols = ["anio","semana","entidad_federativa","enfermedad","sexo"]
    dupes = df.duplicated(key_cols).sum()
    if dupes > 0:
        logger.warning(f"Eliminando {dupes} filas duplicadas antes del reindex")
        df = df.drop_duplicates(subset=key_cols, keep="first")

    anios_validos   = sorted([a for a in df["anio"].unique()   if a > 2000])
    semanas_validas = sorted([s for s in df["semana"].unique() if 1 <= s <= 53])

    idx = pd.MultiIndex.from_product(
        [anios_validos, semanas_validas, ENTIDADES,
         list(DISEASE_CONFIG.keys()), ["H", "M"]],
        names=["anio","semana","entidad_federativa","enfermedad","sexo"]
    )

    df_idx  = df.set_index(key_cols)
    df_full = df_idx.reindex(idx).reset_index()

    for col in ["casos_semana","casos_acumulado","casos_acum_anterior"]:
        df_full[col] = df_full[col].fillna(0).astype(int)

    df_full["cie10"] = df_full["enfermedad"].map(DISEASE_CIE10)

    n_added = len(df_full) - len(df)
    if n_added > 0:
        logger.info(f"Combinaciones rellenadas con 0: {n_added:,}")

    return df_full


# ─────────────────────────────────────────────────────────────
# PASO 2 — NORMALIZAR SEXO
# ─────────────────────────────────────────────────────────────

def _normalize_sexo(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mask_old = (df["anio"] <= 2020) & (df["sexo"].isin(["M", "F"]))
    if mask_old.any():
        df.loc[mask_old & (df["sexo"] == "M"), "sexo"] = "H"
        df.loc[mask_old & (df["sexo"] == "F"), "sexo"] = "M"
        logger.info("Sexo normalizado: M/F → H/M para años ≤ 2020")
    return df


# ─────────────────────────────────────────────────────────────
# PASO 3 — COLUMNAS DERIVADAS
# ─────────────────────────────────────────────────────────────

def _semana_a_fecha(anio: int, semana: int) -> _dt.date:
    """Lunes de la semana epidemiológica ISO."""
    anio, semana = int(anio), int(semana)
    try:
        return _dt.date.fromisocalendar(anio, semana, 1)
    except ValueError:
        return _dt.date.fromisocalendar(anio, 52, 1)


def _add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Temporales ────────────────────────────────────────────
    df["fecha_inicio"] = df.apply(
        lambda r: _semana_a_fecha(r["anio"], r["semana"]), axis=1
    )
    df["semana_año"] = (df["anio"].astype(str) + "-W" +
                        df["semana"].astype(str).str.zfill(2))

    # ── Agregados H+M ─────────────────────────────────────────
    key = ["anio","semana","entidad_federativa","enfermedad"]
    agg = (df.groupby(key)[["casos_acumulado","casos_semana",
                             "casos_acum_anterior"]]
             .sum()
             .rename(columns={
                 "casos_acumulado"    : "casos_ambos_sexos",
                 "casos_semana"       : "casos_sem_total",
                 "casos_acum_anterior": "casos_ant_total",
             })
             .reset_index())
    df = df.merge(agg, on=key, how="left")

    # ── Tasa de crecimiento ───────────────────────────────────
    df["tasa_crecimiento"] = np.where(
        df["casos_ant_total"] > 0,
        df["casos_ambos_sexos"] / df["casos_ant_total"],
        np.nan
    )
    df["incr_semanal"] = df["casos_ambos_sexos"] - df["casos_ant_total"]

    # ── Ciclicidad ────────────────────────────────────────────
    df["semana_sin"] = np.sin(2 * np.pi * df["semana"] / 52)
    df["semana_cos"] = np.cos(2 * np.pi * df["semana"] / 52)

    # ── Flag brote ────────────────────────────────────────────
    df["es_brote"] = df["casos_acumulado"] > 0

    # ── Casos nuevos por diferencia de acumulado ──────────────
    # Más confiable que casos_semana para series de tiempo
    # porque está desglosado por H/M e incorpora correcciones retroactivas
    df = df.sort_values(
        ["enfermedad","entidad_federativa","sexo","anio","semana"]
    )
    df["casos_nuevos"] = (
        df.groupby(["enfermedad","entidad_federativa","sexo","anio"])
          ["casos_acumulado"]
          .diff()
          .fillna(df["casos_semana"])   # semana 1 del año → usar casos_semana
          .clip(lower=0)                # evitar negativos por correcciones
          .astype(int)
    )

    # Versión H+M combinada para modelo univariado
    nuevos_agg = (df.groupby(key)["casos_nuevos"]
                    .sum()
                    .rename("casos_nuevos_total")
                    .reset_index())
    df = df.merge(nuevos_agg, on=key, how="left")

    return df


# ─────────────────────────────────────────────────────────────
# PASO 4 — ORDENAR Y TIPAR
# ─────────────────────────────────────────────────────────────

def _finalize(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(
        ["enfermedad","entidad_federativa","sexo","anio","semana"]
    ).reset_index(drop=True)

    int_cols = [
        "anio","semana",
        "casos_semana","casos_acumulado","casos_acum_anterior",
        "casos_ambos_sexos","casos_sem_total","casos_ant_total",
        "incr_semanal","casos_nuevos","casos_nuevos_total",
    ]
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    float_cols = ["tasa_crecimiento","semana_sin","semana_cos"]
    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    df["fecha_inicio"] = pd.to_datetime(df["fecha_inicio"])
    df["es_brote"]     = df["es_brote"].astype(bool)
    df["sexo"]         = df["sexo"].astype("category")
    df["enfermedad"]   = df["enfermedad"].astype("category")

    assert df["anio"].between(2015, 2027).all(),   "Años fuera de rango"
    assert df["semana"].between(1, 53).all(),       "Semanas fuera de rango"
    assert set(df["sexo"].cat.categories) == {"H","M"}, \
        f"Sexos inesperados: {df['sexo'].unique()}"

    logger.info(f"DataFrame final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    return df


# ─────────────────────────────────────────────────────────────
# FUNCIÓN PRINCIPAL
# ─────────────────────────────────────────────────────────────

def transform(df_raw: pd.DataFrame) -> pd.DataFrame:
    print("🔄 Iniciando transformación...")
    n0 = len(df_raw)

    df = _normalize_sexo(df_raw)
    print(f"   1/4 Sexo normalizado")

    df = _fill_missing_combinations(df)
    print(f"   2/4 Combinaciones completadas: {len(df):,} filas "
          f"(+{len(df)-n0:,} rellenos)")

    df = _add_derived_columns(df)
    print(f"   3/4 Columnas derivadas agregadas")

    df = _finalize(df)
    print(f"   4/4 Tipos y orden finalizados")

    print(f"\n✅ Transformación completa: {df.shape[0]:,} filas × {df.shape[1]} cols")
    return df


print("✅ transformer.py v3 cargado")

✅ transformer.py v3 cargado


In [ ]:
# ============================================================
# CELDA 8: loader.py
# Módulo 3b — Guardado a CSV y Parquet
# ============================================================

import os
import pandas as pd

def save(df: pd.DataFrame,
         name: str = "boletin_epidemiologico",
         out_dir: str = None) -> dict:
    """
    Guarda el DataFrame en CSV y Parquet.
    Retorna dict con rutas de archivos generados.
    """
    out_dir = out_dir or PROCESSED_DIR
    os.makedirs(out_dir, exist_ok=True)

    paths = {}

    # ── CSV ───────────────────────────────────────────────────
    csv_path = os.path.join(out_dir, f"{name}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    csv_mb = os.path.getsize(csv_path) / 1_048_576
    paths["csv"] = csv_path
    print(f"   💾 CSV     → {csv_path}  ({csv_mb:.1f} MB)")

    # ── Parquet ───────────────────────────────────────────────
    parquet_path = os.path.join(out_dir, f"{name}.parquet")
    df.to_parquet(
        parquet_path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
    pq_mb = os.path.getsize(parquet_path) / 1_048_576
    paths["parquet"] = parquet_path
    print(f"   💾 Parquet → {parquet_path}  ({pq_mb:.1f} MB)")

    return paths


def load_parquet(path: str) -> pd.DataFrame:
    """Carga el Parquet procesado para uso en modelos."""
    df = pd.read_parquet(path, engine="pyarrow")
    print(f"✅ Cargado: {df.shape[0]:,} filas × {df.shape[1]} cols")
    print(f"   Rango: {df['anio'].min()}–{df['anio'].max()}")
    print(f"   Enfermedades: {sorted(df['enfermedad'].unique())}")
    return df


print("✅ loader.py cargado")

✅ loader.py cargado


In [ ]:
# ============================================================
# CELDA 9: Pipeline completo — test con sem23/2020
# ============================================================

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)

# ── 1. Extraer ────────────────────────────────────────────────
print("="*55)
print("PASO 1 — EXTRACCIÓN")
print("="*55)
df_raw = extract_pdf("/content/etl_boletin/data/raw/pdfs/2020/sem23.pdf")
print(f"Registros extraídos: {len(df_raw):,}")

# ── 2. Transformar ────────────────────────────────────────────
print("\n" + "="*55)
print("PASO 2 — TRANSFORMACIÓN")
print("="*55)
df = transform(df_raw)

# ── 3. Guardar ────────────────────────────────────────────────
print("\n" + "="*55)
print("PASO 3 — CARGA")
print("="*55)
paths = save(df, name="test_sem23_2020")

# ── 4. Verificar output ───────────────────────────────────────
print("\n" + "="*55)
print("VERIFICACIÓN DEL DATAFRAME FINAL")
print("="*55)

print(f"\nShape       : {df.shape}")
print(f"Columnas    : {list(df.columns)}")
print(f"\nDTypes:")
print(df.dtypes.to_string())

print(f"\nEnfermedades presentes: {sorted(df['enfermedad'].unique())}")
print(f"Sexos presentes       : {sorted(df['sexo'].unique())}")
print(f"Entidades             : {df['entidad_federativa'].nunique()}")
print(f"Rango fechas          : {df['fecha_inicio'].min()} → {df['fecha_inicio'].max()}")

print(f"\nMuestra (Varicela, Sonora):")
mask = (
    (df["enfermedad"] == "Varicela") &
    (df["entidad_federativa"] == "Sonora")
)
cols_show = [
    "anio","semana","fecha_inicio","sexo",
    "casos_semana","casos_acumulado","casos_ambos_sexos",
    "tasa_crecimiento","es_brote"
]
print(df[mask][cols_show].to_string(index=False))

print(f"\nVerificación Sarampión (debe tener 0s para todas las entidades):")
mask_sar = df["enfermedad"] == "Sarampion"
sar_sum  = df[mask_sar]["casos_acumulado"].sum()
sar_rows = df[mask_sar]["entidad_federativa"].nunique()
print(f"  Filas  : {mask_sar.sum()}")
print(f"  Entidades cubiertas: {sar_rows}/32")
print(f"  Total acumulado: {sar_sum} (esperado=0 — sem23/2020 sin brote)")

# ── 5. Recargar desde Parquet ─────────────────────────────────
print(f"\n" + "="*55)
print("VERIFICACIÓN PARQUET — recarga desde disco")
print("="*55)
df_reload = load_parquet(paths["parquet"])
assert df_reload.shape == df.shape, "❌ Shape no coincide tras recarga"
print("✅ Parquet íntegro — shape coincide")

PASO 1 — EXTRACCIÓN
Registros extraídos: 256

PASO 2 — TRANSFORMACIÓN
🔄 Iniciando transformación...
   1/4 Sexo normalizado
   2/4 Combinaciones completadas: 320 filas (+64 rellenos)
   3/4 Columnas derivadas agregadas
   4/4 Tipos y orden finalizados

✅ Transformación completa: 320 filas × 21 cols

PASO 3 — CARGA
   💾 CSV     → /content/etl_boletin/data/processed/test_sem23_2020.csv  (0.0 MB)
   💾 Parquet → /content/etl_boletin/data/processed/test_sem23_2020.parquet  (0.0 MB)

VERIFICACIÓN DEL DATAFRAME FINAL

Shape       : (320, 21)
Columnas    : ['anio', 'semana', 'entidad_federativa', 'enfermedad', 'sexo', 'cie10', 'casos_semana', 'casos_acumulado', 'casos_acum_anterior', 'fecha_inicio', 'semana_año', 'casos_ambos_sexos', 'casos_sem_total', 'casos_ant_total', 'tasa_crecimiento', 'incr_semanal', 'semana_sin', 'semana_cos', 'es_brote', 'casos_nuevos', 'casos_nuevos_total']

DTypes:
anio                            int64
semana                          int64
entidad_federativa         

In [ ]:
# ============================================================
# CELDA 10: main.py
# Orquestador completo — descarga, extrae, transforma y guarda
# ============================================================

import os
import time
import logging
import pandas as pd
from pathlib import Path
from datetime import datetime

# ── Logging a archivo + consola ───────────────────────────────
os.makedirs(LOG_DIR, exist_ok=True)

log_file = os.path.join(
    LOG_DIR, f"etl_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(),
    ]
)
logger = logging.getLogger("main")


# ─────────────────────────────────────────────────────────────
# REPORTE DE CALIDAD
# ─────────────────────────────────────────────────────────────

def quality_report(df: pd.DataFrame) -> dict:
    report = {}

    cobertura = (df.groupby(["anio", "enfermedad"])["entidad_federativa"]
                   .nunique()
                   .rename("entidades_con_datos")
                   .reset_index())
    report["cobertura"] = cobertura

    sems_por_año = (df.groupby("anio")["semana"]
                      .nunique()
                      .rename("semanas")
                      .reset_index())
    report["semanas_por_año"] = sems_por_año

    enf_requeridas = [
        n for n, c in DISEASE_CONFIG.items()
        if not c.get("optional", False)
    ]
    enf_por_sem = (df[df["enfermedad"].isin(enf_requeridas)]
                     .groupby(["anio", "semana"])["enfermedad"]
                     .nunique()
                     .reset_index())
    incompletas = enf_por_sem[
        enf_por_sem["enfermedad"] < len(enf_requeridas)
    ]
    report["semanas_incompletas"] = incompletas

    resumen = (df.groupby(["anio", "enfermedad"])["casos_ambos_sexos"]
                 .max()
                 .unstack("enfermedad")
                 .fillna(0)
                 .astype(int))
    report["resumen_anual"] = resumen

    return report


def print_quality_report(report: dict):
    print("\n" + "="*60)
    print("REPORTE DE CALIDAD")
    print("="*60)

    print("\n── Semanas procesadas por año ──")
    print(report["semanas_por_año"].to_string(index=False))

    print("\n── Semanas con enfermedades faltantes ──")
    inc = report["semanas_incompletas"]
    if inc.empty:
        print("   ✅ Ninguna — cobertura completa")
    else:
        print(f"   ⚠️  {len(inc)} semanas incompletas:")
        print(inc.to_string(index=False))

    print("\n── Casos acumulados máximos por año y enfermedad ──")
    print(report["resumen_anual"].to_string())


# ─────────────────────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ─────────────────────────────────────────────────────────────

def run_etl(
    years_zip      : list = None,
    years_pdf      : list = None,   # ← CAMBIO: default ahora es YEARS_PDF
    force_download : bool = False,
    output_name    : str  = "boletin_epidemiologico",
) -> pd.DataFrame:
    """
    Pipeline completo ETL.

    Parámetros:
      years_zip      : años a descargar como ZIP (default: YEARS_ZIP)
      years_pdf      : años con PDFs sueltos     (default: YEARS_PDF)
      force_download : re-descarga aunque el archivo ya exista
      output_name    : nombre base del archivo de salida
    """
    t_start   = time.time()
    years_zip = years_zip if years_zip is not None else YEARS_ZIP
    years_pdf = years_pdf if years_pdf is not None else YEARS_PDF  # ← CAMBIO

    logger.info("="*55)
    logger.info("INICIO ETL — Boletín Epidemiológico DGE")
    logger.info(f"Años ZIP : {years_zip[0]} → {years_zip[-1]}" if years_zip else "Años ZIP: ninguno")
    logger.info(f"Años PDF : {years_pdf or 'ninguno'}")
    logger.info("="*55)

    # ── PASO 1: Descarga ──────────────────────────────────────
    print("\n" + "="*55)
    print("PASO 1 — DESCARGA")
    print("="*55)

    if force_download:
        # Eliminar ZIPs para forzar re-descarga
        for year in years_zip:
            zip_path = os.path.join(RAW_ZIP_DIR, f"{year}.zip")
            if os.path.exists(zip_path):
                os.remove(zip_path)
                logger.info(f"ZIP eliminado: {zip_path}")

        # Eliminar PDFs sueltos para forzar re-descarga
        for year in years_pdf:
            year_dir = os.path.join(RAW_PDF_DIR, str(year))
            if os.path.isdir(year_dir):
                import shutil
                shutil.rmtree(year_dir)
                logger.info(f"Dir eliminado: {year_dir}")

    # ZIPs 2016-2024
    if years_zip:
        download_zips(years=years_zip)

    # PDFs sueltos 2025-2026                    ← CAMBIO: bloque nuevo
    if years_pdf:
        urls_configuradas = sum(
            len(PDF_URLS_BY_YEAR.get(y, {}))
            for y in years_pdf
        )
        if urls_configuradas == 0:
            print(f"\n⚠️  AÑOS PDF {years_pdf}: no hay URLs en PDF_URLS_BY_YEAR")
            print("   Agrega las URLs en la Celda 2 (config.py) y vuelve a correr")
        else:
            download_pdfs_sueltos(years=years_pdf)

    # ── PASO 2: Inventario ────────────────────────────────────
    print("\n" + "="*55)
    print("PASO 2 — INVENTARIO DE PDFs")
    print("="*55)

    all_years = years_zip + years_pdf
    inventory = get_pdf_inventory(years=all_years)
    print_inventory(inventory)

    total_pdfs = sum(len(v) for v in inventory.values())
    if total_pdfs == 0:
        logger.error("No se encontraron PDFs — abortando")
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    # ── PASO 3: Extracción ────────────────────────────────────
    print("\n" + "="*55)
    print("PASO 3 — EXTRACCIÓN DE TABLAS")
    print("="*55)

    df_raw = extract_all_pdfs(inventory=inventory, years=all_years)

    if df_raw.empty:
        logger.error("Extracción vacía — abortando")
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    # Checkpoint raw
    raw_path = os.path.join(PROCESSED_DIR, f"{output_name}_raw.parquet")
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    df_raw.to_parquet(raw_path, index=False, engine="pyarrow")
    logger.info(f"Checkpoint raw: {raw_path}")

    # ── PASO 4: Transformación ────────────────────────────────
    print("\n" + "="*55)
    print("PASO 4 — TRANSFORMACIÓN")
    print("="*55)

    df = transform(df_raw)

    # ── PASO 5: Carga ─────────────────────────────────────────
    print("\n" + "="*55)
    print("PASO 5 — GUARDADO")
    print("="*55)

    paths = save(df, name=output_name)

    # ── PASO 6: Reporte de calidad ────────────────────────────
    print("\n" + "="*55)
    print("PASO 6 — REPORTE DE CALIDAD")
    print("="*55)

    report = quality_report(df)
    print_quality_report(report)

    # ── Resumen final ─────────────────────────────────────────
    t_end    = time.time()
    mins, seg = divmod(int(t_end - t_start), 60)

    # Desglose de PDFs por fuente                ← CAMBIO: más detalle
    pdfs_zip = sum(len(inventory.get(y, [])) for y in years_zip)
    pdfs_pdf = sum(len(inventory.get(y, [])) for y in years_pdf)

    print("\n" + "="*55)
    print("RESUMEN FINAL")
    print("="*55)
    print(f"   PDFs ZIP (2016-2024) : {pdfs_zip:,}")
    print(f"   PDFs sueltos (2025+) : {pdfs_pdf:,}")
    print(f"   PDFs totales         : {total_pdfs:,}")
    print(f"   Registros totales    : {len(df):,}")
    print(f"   Años cubiertos       : {sorted(df['anio'].unique())}")
    print(f"   Columnas             : {df.shape[1]}")
    print(f"   CSV                  : {paths['csv']}")
    print(f"   Parquet              : {paths['parquet']}")
    print(f"   Log                  : {log_file}")
    print(f"   Duración total       : {mins}m {seg}s")
    print("="*55)

    logger.info(f"ETL completado en {mins}m {seg}s — {len(df):,} registros")
    return df


print("✅ main.py cargado")

✅ main.py cargado


In [ ]:
# ============================================================
# CELDA 11: Test rápido con un solo año
# ============================================================

# Silenciar pdfminer antes de extraer
import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)
logging.getLogger("pdfplumber").setLevel(logging.ERROR)

# Test con 2020 solamente
inv_test = get_pdf_inventory(years=[2020])
print(f"PDFs 2020: {len(inv_test.get(2020, []))}")

df_raw_test   = extract_all_pdfs(inventory=inv_test, years=[2020])
df_final_test = transform(df_raw_test)
paths_test    = save(df_final_test, name="test_2020_completo")

report = quality_report(df_final_test)
print_quality_report(report)

PDFs 2020: 53

🔬 Extrayendo datos de 53 PDFs...


100%|██████████| 53/53 [11:33<00:00, 13.09s/PDF, año=2020, ok=53]


   2020: 13,568 registros

✅ Total: 13,568 registros
   Enfermedades : ['Erisipela', 'Escarlatina', 'Rubeola', 'Varicela']
   Años         : [np.int64(2019), np.int64(2020)]
🔄 Iniciando transformación...
   1/4 Sexo normalizado
   2/4 Combinaciones completadas: 33,920 filas (+20,352 rellenos)
   3/4 Columnas derivadas agregadas
   4/4 Tipos y orden finalizados

✅ Transformación completa: 33,920 filas × 21 cols
   💾 CSV     → /content/etl_boletin/data/processed/test_2020_completo.csv  (4.0 MB)
   💾 Parquet → /content/etl_boletin/data/processed/test_2020_completo.parquet  (0.1 MB)

REPORTE DE CALIDAD

── Semanas procesadas por año ──
 anio  semanas
 2019       53
 2020       53

── Semanas con enfermedades faltantes ──
   ✅ Ninguna — cobertura completa

── Casos acumulados máximos por año y enfermedad ──
enfermedad  Erisipela  Escarlatina  Rubeola  Sarampion  Varicela
anio                                                            
2019              961          699        0          0  

/tmp/ipykernel_3158/3581854920.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cobertura = (df.groupby(["anio", "enfermedad"])["entidad_federativa"]
/tmp/ipykernel_3158/3581854920.py:63: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumen = (df.groupby(["anio", "enfermedad"])["casos_ambos_sexos"]


In [ ]:
# ============================================================
# DIAGNÓSTICO: estructura de carpetas de PDFs
# ============================================================
from pathlib import Path
from collections import Counter

for year in [2016, 2017, 2018, 2019, 2021, 2022]:
    year_dir = f"/content/etl_boletin/data/raw/pdfs/{year}"
    all_pdfs = list(Path(year_dir).rglob("*.pdf"))

    print(f"\n{year} — {len(all_pdfs)} PDFs")
    print("Subcarpetas:")
    dirs = set(str(p.parent) for p in all_pdfs)
    for d in sorted(dirs):
        n = len(list(Path(d).glob("*.pdf")))
        print(f"  {d}  →  {n} PDFs")

    # Nombres duplicados
    nombres = [p.name for p in all_pdfs]
    dupes = {k:v for k,v in Counter(nombres).items() if v > 1}
    print(f"Nombres duplicados: {len(dupes)}")
    if dupes:
        for nombre, count in list(dupes.items())[:3]:
            print(f"  '{nombre}' aparece {count} veces")


2016 — 52 PDFs
Subcarpetas:
  /content/etl_boletin/data/raw/pdfs/2016  →  52 PDFs
Nombres duplicados: 0

2017 — 52 PDFs
Subcarpetas:
  /content/etl_boletin/data/raw/pdfs/2017  →  52 PDFs
Nombres duplicados: 0

2018 — 52 PDFs
Subcarpetas:
  /content/etl_boletin/data/raw/pdfs/2018  →  52 PDFs
Nombres duplicados: 0

2019 — 52 PDFs
Subcarpetas:
  /content/etl_boletin/data/raw/pdfs/2019  →  52 PDFs
Nombres duplicados: 0

2021 — 52 PDFs
Subcarpetas:
  /content/etl_boletin/data/raw/pdfs/2021  →  52 PDFs
Nombres duplicados: 0

2022 — 52 PDFs
Subcarpetas:
  /content/etl_boletin/data/raw/pdfs/2022  →  52 PDFs
Nombres duplicados: 0


In [ ]:
# ============================================================
# CELDA 11: ETL con procesamiento paralelo (CPU)
# ============================================================
import logging
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# Silenciar pdfminer
logging.getLogger("pdfminer").setLevel(logging.ERROR)
logging.getLogger("pdfplumber").setLevel(logging.ERROR)

# Colab T4 tiene 2 CPUs disponibles
N_WORKERS = multiprocessing.cpu_count()
print(f"CPUs disponibles: {N_WORKERS}")


def _extract_pdf_worker(pdf_path: str) -> pd.DataFrame:
    """Worker para ejecución en proceso paralelo."""
    import logging
    logging.getLogger("pdfminer").setLevel(logging.ERROR)
    logging.getLogger("pdfplumber").setLevel(logging.ERROR)
    return extract_pdf(pdf_path)


def extract_all_pdfs_parallel(inventory: dict,
                               years: list = None,
                               n_workers: int = None) -> pd.DataFrame:
    """
    Versión paralela de extract_all_pdfs.
    Usa ProcessPoolExecutor para procesar múltiples PDFs simultáneamente.
    """
    years     = years or sorted(inventory.keys())
    n_workers = n_workers or N_WORKERS

    # Aplanar todos los PDFs en una sola lista
    all_pdfs = []
    for year in years:
        for pdf in sorted(set(inventory.get(year, []))):
            all_pdfs.append(pdf)

    total = len(all_pdfs)
    print(f"\n🔬 Extrayendo {total} PDFs con {n_workers} workers en paralelo...")

    frames = []
    errores = []

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {
            executor.submit(_extract_pdf_worker, pdf): pdf
            for pdf in all_pdfs
        }
        with tqdm(total=total, unit="PDF") as pbar:
            for future in as_completed(futures):
                pdf_path = futures[future]
                try:
                    df = future.result()
                    if not df.empty:
                        frames.append(df)
                except Exception as e:
                    errores.append(pdf_path)
                    logging.getLogger("main").error(
                        f"Error en worker: {pdf_path} — {e}"
                    )
                pbar.update(1)

    if errores:
        print(f"\n⚠️  PDFs con error: {len(errores)}")
        for e in errores[:5]:
            print(f"   {e}")

    if not frames:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    df_final = (pd.concat(frames, ignore_index=True)
                .sort_values(["anio","semana","enfermedad",
                              "entidad_federativa","sexo"])
                .reset_index(drop=True))

    print(f"\n✅ Total: {len(df_final):,} registros")
    print(f"   Enfermedades : {sorted(df_final['enfermedad'].unique())}")
    print(f"   Años         : {sorted(df_final['anio'].unique())}")
    return df_final


# ── Ejecutar ETL paralelo ─────────────────────────────────────
inventory = get_pdf_inventory(years=YEARS_ZIP + YEARS_PDF)
print_inventory(inventory)

df_raw   = extract_all_pdfs_parallel(inventory, years=YEARS_ZIP + YEARS_PDF)
df_final = transform(df_raw)
paths    = save(df_final, name="boletin_epidemiologico_2016_2026")

report = quality_report(df_final)
print_quality_report(report)

CPUs disponibles: 2

📋 Inventario de PDFs:
Año      PDFs     Fuente
------------------------------
2016     52       ZIP
2017     52       ZIP
2018     52       ZIP
2019     52       ZIP
2020     53       ZIP
2021     52       ZIP
2022     52       ZIP
2023     53       ZIP
2024     52       ZIP
2025     53       PDF suelto
2026     12       PDF suelto
------------------------------
TOTAL    535     

🔬 Extrayendo 535 PDFs con 2 workers en paralelo...


100%|██████████| 535/535 [1:54:13<00:00, 12.81s/PDF]



✅ Total: 139,992 registros
   Enfermedades : ['Erisipela', 'Escarlatina', 'Rubeola', 'Sarampion', 'Varicela']
   Años         : [np.int64(0), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
🔄 Iniciando transformación...
   1/4 Sexo normalizado
   2/4 Combinaciones completadas: 186,560 filas (+46,568 rellenos)
   3/4 Columnas derivadas agregadas
   4/4 Tipos y orden finalizados

✅ Transformación completa: 186,560 filas × 21 cols
   💾 CSV     → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv  (22.9 MB)
   💾 Parquet → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet  (1.1 MB)

REPORTE DE CALIDAD

── Semanas procesadas por año ──
 anio  semanas
 2016       53
 2017       53
 2018       53
 2019       53
 2020       53
 2021       53
 2022       53
 2023       53
 2024       53
 2025       53
 2026       53

── 

/tmp/ipykernel_3158/3581854920.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cobertura = (df.groupby(["anio", "enfermedad"])["entidad_federativa"]
/tmp/ipykernel_3158/3581854920.py:63: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumen = (df.groupby(["anio", "enfermedad"])["casos_ambos_sexos"]


In [ ]:
# ============================================================
# CORRECCIÓN: casos_nuevos y casos_nuevos_total
# Se aplica directamente sobre el CSV/Parquet ya generado
# SIN necesidad de re-correr el ETL completo
# ============================================================

import pandas as pd
import numpy as np

# ── Cargar el CSV ya generado ─────────────────────────────────
print("Cargando CSV...")
df = pd.read_csv(
    "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv"
)
print(f"Shape original: {df.shape}")

# ── Recalcular casos_nuevos correctamente ─────────────────────
print("Recalculando casos_nuevos...")

df = df.sort_values(
    ["enfermedad","entidad_federativa","sexo","anio","semana"]
).reset_index(drop=True)

# diff del acumulado por grupo (enfermedad/estado/sexo/año)
df["casos_nuevos"] = (
    df.groupby(["enfermedad","entidad_federativa","sexo","anio"])
      ["casos_acumulado"]
      .diff()
      .clip(lower=0)
      .astype("Int64")   # permite NaN temporalmente
)

# Semana 1: diff es NaN → usar casos_semana / 2
# (casos_semana del PDF es H+M sin desglose, dividir entre 2 por sexo)
df["casos_nuevos"] = df["casos_nuevos"].fillna(
    (df["casos_semana"] / 2).round().astype(int)
).astype(int)

# ── Recalcular casos_nuevos_total (H+M) ──────────────────────
print("Recalculando casos_nuevos_total...")

key = ["anio","semana","entidad_federativa","enfermedad"]
nuevos_agg = (
    df.groupby(key)["casos_nuevos"]
      .sum()
      .rename("casos_nuevos_total")
      .reset_index()
)

# Eliminar columnas viejas y unir las nuevas
df = df.drop(columns=["casos_nuevos","casos_nuevos_total"], errors="ignore")
df = df.merge(nuevos_agg, on=key, how="left")

# Agregar casos_nuevos individual desde el total ya calculado
# casos_nuevos individual = diff del acumulado (ya calculado arriba)
df["casos_nuevos"] = (
    df.groupby(["enfermedad","entidad_federativa","sexo","anio"])
      ["casos_acumulado"]
      .diff()
      .clip(lower=0)
      .fillna((df["casos_semana"] / 2).round())
      .astype(int)
)

# ── Verificación rápida ───────────────────────────────────────
print("\nVerificación semana 1 (Varicela, Sonora, H, 2016):")
mask = (
    (df["enfermedad"]=="Varicela") &
    (df["entidad_federativa"]=="Sonora") &
    (df["sexo"]=="H") &
    (df["anio"]==2016)
)
check = df[mask].sort_values("semana")[
    ["semana","casos_acumulado","casos_nuevos","casos_nuevos_total"]
].head(6)
print(check.to_string(index=False))

# semana 1: casos_nuevos_total debe ser ~40 (no 80)
sem1_ok = df[
    (df["semana"]==1) &
    (df["enfermedad"]=="Varicela") &
    (df["entidad_federativa"]=="Sonora") &
    (df["sexo"]=="H") &
    (df["anio"]==2016)
]["casos_nuevos_total"].values[0]

print(f"\nSemana 1 Varicela Sonora: {sem1_ok} (esperado ~40) "
      f"{'✅' if sem1_ok <= 50 else '❌'}")

neg = (df["casos_nuevos"] < 0).sum()
nulos = df["casos_nuevos"].isna().sum()
print(f"Valores negativos: {neg} ✅" if neg==0 else f"Valores negativos: {neg} ❌")
print(f"Nulos: {nulos} ✅" if nulos==0 else f"Nulos: {nulos} ❌")

print(f"\nShape final: {df.shape}")

# ── Guardar versión corregida ─────────────────────────────────
print("\nGuardando...")

csv_path     = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv"
parquet_path = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_parquet(parquet_path, index=False, engine="pyarrow", compression="snappy")

csv_mb     = os.path.getsize(csv_path)     / 1_048_576
parquet_mb = os.path.getsize(parquet_path) / 1_048_576

print(f"✅ CSV     → {csv_path} ({csv_mb:.1f} MB)")
print(f"✅ Parquet → {parquet_path} ({parquet_mb:.1f} MB)")
print("\n✅ Corrección aplicada sin re-correr el ETL")

Cargando CSV...
Shape original: (186560, 21)
Recalculando casos_nuevos...
Recalculando casos_nuevos_total...

Verificación semana 1 (Varicela, Sonora, H, 2016):
 semana  casos_acumulado  casos_nuevos  casos_nuevos_total
      1                2            20                  40
      2               36            34                  34
      3               73            37                  91
      4              102            29                  64
      5              150            48                  83
      6              182            32                  70

Semana 1 Varicela Sonora: 40 (esperado ~40) ✅
Valores negativos: 0 ✅
Nulos: 0 ✅

Shape final: (186560, 21)

Guardando...
✅ CSV     → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv (22.8 MB)
✅ Parquet → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet (1.1 MB)

✅ Corrección aplicada sin re-correr el ETL


In [ ]:
# ============================================================
# CELDA: Forzar reprocesamiento completo de 2024
# ============================================================

import os
import re
import logging
import pandas as pd
import numpy as np
from pathlib import Path

logging.getLogger("pdfminer").setLevel(logging.ERROR)
logging.getLogger("pdfplumber").setLevel(logging.ERROR)

CSV_ACTUAL = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv"
CSV_SALIDA = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv"
PQ_SALIDA  = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet"

# ── PASO 1: Cargar CSV y eliminar TODO 2024 ───────────────────
print("="*55)
print("PASO 1 — ELIMINAR 2024 DEL CSV ACTUAL")
print("="*55)

df_actual = pd.read_csv(CSV_ACTUAL)
print(f"Filas antes : {len(df_actual):,}")

df_sin_2024 = df_actual[df_actual["anio"] != 2024].copy()
print(f"Filas sin 2024: {len(df_sin_2024):,}")
print(f"Filas eliminadas: {len(df_actual) - len(df_sin_2024):,}")

# ── PASO 2: Extraer 2024 completo desde PDFs ─────────────────
print("\n" + "="*55)
print("PASO 2 — EXTRACCIÓN 2024 COMPLETO")
print("="*55)

inv_2024  = get_pdf_inventory(years=[2024])
print(f"PDFs disponibles: {len(inv_2024.get(2024, []))}")

df_raw_2024 = extract_all_pdfs(inventory=inv_2024, years=[2024])
print(f"Registros extraídos: {len(df_raw_2024):,}")

# ── PASO 3: Transformar ───────────────────────────────────────
print("\n" + "="*55)
print("PASO 3 — TRANSFORMACIÓN")
print("="*55)

df_2024 = transform(df_raw_2024)
print(f"Registros transformados: {len(df_2024):,}")

# Verificar cobertura
sems_ok = sorted(df_2024[df_2024["casos_acumulado"]>0]["semana"].unique())
print(f"Semanas con datos: {len(sems_ok)} → {sems_ok}")

# ── PASO 4: Fusionar ──────────────────────────────────────────
print("\n" + "="*55)
print("PASO 4 — FUSIÓN")
print("="*55)

df_fusion = (
    pd.concat([df_sin_2024, df_2024], ignore_index=True)
      .sort_values(["anio","semana","enfermedad",
                    "entidad_federativa","sexo"])
      .reset_index(drop=True)
)
print(f"Filas finales: {len(df_fusion):,}")

# ── PASO 5: Verificar 2024 ────────────────────────────────────
print("\n" + "="*55)
print("PASO 5 — VERIFICACIÓN 2024")
print("="*55)

sems_finales = (df_fusion[
    (df_fusion["anio"]==2024) &
    (df_fusion["sexo"]=="H")
].groupby("semana")["casos_acumulado"].sum().reset_index())

sems_finales["tiene_datos"] = sems_finales["casos_acumulado"] > 0
con_datos    = sems_finales["tiene_datos"].sum()
sin_datos    = (~sems_finales["tiene_datos"]).sum()

print(f"Semanas con datos : {con_datos}")
print(f"Semanas sin datos : {sin_datos}")

if sin_datos > 0:
    vacias = sems_finales[~sems_finales["tiene_datos"]]["semana"].tolist()
    print(f"Semanas vacías    : {vacias}")

# ── PASO 6: Guardar ───────────────────────────────────────────
print("\n" + "="*55)
print("PASO 6 — GUARDADO")
print("="*55)

df_fusion.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
df_fusion.to_parquet(
    PQ_SALIDA, index=False,
    engine="pyarrow", compression="snappy"
)

csv_mb = os.path.getsize(CSV_SALIDA) / 1_048_576
pq_mb  = os.path.getsize(PQ_SALIDA)  / 1_048_576

print(f"✅ CSV     → {CSV_SALIDA} ({csv_mb:.1f} MB)")
print(f"✅ Parquet → {PQ_SALIDA}  ({pq_mb:.1f} MB)")
print(f"\n✅ 2024 reprocesado — {len(df_fusion):,} filas totales")

PASO 1 — ELIMINAR 2024 DEL CSV ACTUAL
Filas antes : 186,560
Filas sin 2024: 169,600
Filas eliminadas: 16,960

PASO 2 — EXTRACCIÓN 2024 COMPLETO
PDFs disponibles: 52

🔬 Extrayendo datos de 52 PDFs...


100%|██████████| 52/52 [13:38<00:00, 15.75s/PDF, año=2024, ok=52]


   2024: 13,312 registros

✅ Total: 13,312 registros
   Enfermedades : ['Erisipela', 'Escarlatina', 'Rubeola', 'Varicela']
   Años         : [np.int64(0), np.int64(2023), np.int64(2024)]
Registros extraídos: 13,312

PASO 3 — TRANSFORMACIÓN
🔄 Iniciando transformación...
   1/4 Sexo normalizado
   2/4 Combinaciones completadas: 33,280 filas (+19,968 rellenos)
   3/4 Columnas derivadas agregadas
   4/4 Tipos y orden finalizados

✅ Transformación completa: 33,280 filas × 21 cols
Registros transformados: 33,280
Semanas con datos: 46 → [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(3

ArrowTypeError: ("Expected bytes, got a 'Timestamp' object", 'Conversion failed for column fecha_inicio with type object')

In [ ]:
# ── PASO 6: Guardar ───────────────────────────────────────────
print("\n" + "="*55)
print("PASO 6 — GUARDADO")
print("="*55)

# Convertir fecha_inicio a datetime antes de guardar en Parquet
df_fusion["fecha_inicio"] = pd.to_datetime(df_fusion["fecha_inicio"])

# Asegurar tipos correctos en columnas categóricas
df_fusion["enfermedad"] = df_fusion["enfermedad"].astype(str)
df_fusion["sexo"]       = df_fusion["sexo"].astype(str)

df_fusion.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
df_fusion.to_parquet(
    PQ_SALIDA, index=False,
    engine="pyarrow", compression="snappy"
)

csv_mb = os.path.getsize(CSV_SALIDA) / 1_048_576
pq_mb  = os.path.getsize(PQ_SALIDA)  / 1_048_576

print(f"✅ CSV     → {CSV_SALIDA} ({csv_mb:.1f} MB)")
print(f"✅ Parquet → {PQ_SALIDA}  ({pq_mb:.1f} MB)")
print(f"\n✅ 2024 reprocesado — {len(df_fusion):,} filas totales")


PASO 6 — GUARDADO
✅ CSV     → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv (24.6 MB)
✅ Parquet → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet  (1.4 MB)

✅ 2024 reprocesado — 202,880 filas totales


In [ ]:
# ============================================================
# CORRECCIÓN FINAL: eliminar duplicados de 2023
# ============================================================
import pandas as pd
import os

CSV_ACTUAL = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv"
CSV_SALIDA = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv"
PQ_SALIDA  = "/content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet"

df = pd.read_csv(CSV_ACTUAL)
print(f"Shape antes : {df.shape}")
print(f"Duplicados  : {df.duplicated(['anio','semana','entidad_federativa','enfermedad','sexo']).sum():,}")

# Deduplicar — conservar primera ocurrencia
key = ["anio","semana","entidad_federativa","enfermedad","sexo"]
df = df.drop_duplicates(subset=key, keep="first").reset_index(drop=True)

print(f"Shape después: {df.shape}")
print(f"\nFilas por año:")
print(df.groupby("anio").size().to_string())

# Verificar
esperado = 186560  # 2016-2026 sin duplicados
ok = "✅" if len(df) == esperado else f"⚠️  esperado ~{esperado}"
print(f"\nTotal filas: {len(df):,} {ok}")

# Guardar
df["fecha_inicio"] = pd.to_datetime(df["fecha_inicio"])
df["enfermedad"]   = df["enfermedad"].astype(str)
df["sexo"]         = df["sexo"].astype(str)

df.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
df.to_parquet(PQ_SALIDA, index=False, engine="pyarrow", compression="snappy")

csv_mb = os.path.getsize(CSV_SALIDA) / 1_048_576
pq_mb  = os.path.getsize(PQ_SALIDA)  / 1_048_576
print(f"\n✅ CSV     → {CSV_SALIDA} ({csv_mb:.1f} MB)")
print(f"✅ Parquet → {PQ_SALIDA}  ({pq_mb:.1f} MB)")

Shape antes : (202880, 21)
Duplicados  : 16,640
Shape después: (186240, 21)

Filas por año:
anio
2016    16960
2017    16960
2018    16960
2019    16960
2020    16960
2021    16960
2022    16960
2023    16960
2024    16640
2025    16960
2026    16960

Total filas: 186,240 ⚠️  esperado ~186560

✅ CSV     → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.csv (22.8 MB)
✅ Parquet → /content/etl_boletin/data/processed/boletin_epidemiologico_2016_2026.parquet  (1.4 MB)
